# Notebook 5: Interactive visualization of subgraphs

Import the Python library dependencies.

In [1]:
from yfiles_jupyter_graphs_for_kuzu import KuzuGraphWidget
import kuzu
import watermark

Run a "watermark" to show which library versions are used in this notebook's runtime environment.

In [2]:
%load_ext watermark
%watermark
%watermark --iversions

Last updated: 2025-11-11T23:07:24.206175-08:00

Python implementation: CPython
Python version       : 3.13.8
IPython version      : 9.1.0

Compiler    : Clang 17.0.0 (clang-1700.0.13.3)
OS          : Darwin
Release     : 24.6.0
Machine     : arm64
Processor   : arm
CPU cores   : 14
Architecture: 64bit

kuzu                          : 0.9.0
yfiles_jupyter_graphs_for_kuzu: 0.0.4
watermark                     : 2.5.0



## Interactive Visualization

Reconnect to the existing graph database.

In [3]:
DB_PATH: str = "./db"

db: kuzu.Database = kuzu.Database(DB_PATH)
conn: kuzu.Connection = kuzu.Connection(db)

Create a [`yFiles`](https://www.yworks.com/products/yfiles-graphs-for-jupyter) graph widget to explore the graph which has just been created.
Define a function to adjust node and edge configurations, to help clarify the interactive visualization.

In [4]:
def node_color (
    node
    ) -> str:
    match node["properties"]["class"]:
        case "sz:Person":
            return "Red"
        case "sz:Organization":
            return "Orange"
        case "sz:DataRecord":
            return "Purple"
        case _:
            return "Black"

In [5]:
g: KuzuGraphWidget = KuzuGraphWidget(conn)

g.add_node_configuration("Entity", color = node_color, text = lambda node: {"text": node["properties"]["descrip"]})
g.add_node_configuration("OpenSanctions", color = "yellow", text = lambda node: {"text": node["properties"]["descrip"]})
g.add_node_configuration("OpenOwnership", color = "green", text = lambda node: {"text": node["properties"]["descrip"]})
g.add_node_configuration("Risk", color = "blue", text = lambda node: {"text": node["properties"]["topic"]})

g.add_relationship_configuration("Related", color = "blue", text = lambda edge: {"text": edge["properties"]["sem_rel"]})

Run interactive visualization using `yFiles` to examine a subgraph describing the [2021 South London Papa Johns](https://www.newsshopper.co.uk/news/19164815.boss-bromley-catford-papa-johns-stores-jailed/) tax evasion case.

In [6]:
query: str = """
MATCH (a)-[b]->(c:Entity)-[d *1..5]->(e)
WHERE c.descrip CONTAINS "Abassin"
RETURN * LIMIT 200;
"""

g.show_cypher(
    query,
    layout = "radial"
)

GraphWidget(layout=Layout(height='500px', width='100%'))

Let's see how the same results look as a dataframe.

In [7]:
res = conn.execute(query)
res.get_as_pl()

a,c,b,e,d
struct[9],struct[5],struct[9],struct[9],struct[2]
"{{126,0},""Entity"",""sz:2"",""LMAR GB LTD"",""sz:Organization"",null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person""}","{{126,0},{4,0},""Related"",{501,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(:DIRECTORSHIP)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,""corp.disqual""}","{[{{126,0},""Entity"",null,""sz:2"",""LMAR GB LTD"",""sz:Organization"",null,null,null}, {{198,0},""Entity"",null,""sz:99"",""WELLHANCIA HEALTH CARE LTD"",""sz:Organization"",null,null,null}, … {{3,2},""OpenSanctions"",null,""sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ"",""Abassin BADSHAH"",""sz:Person"",""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",""https://www.opensanctions.org/entities/NK-25vyVFzt8vdJGgAXMRTwTJ"",null}],[{{4,0},{126,0},""Related"",{663,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(DIRECTORSHIP:)-RECORD_TYPE"",null,null}, {{126,0},{198,0},""Related"",{83,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+REGISTRATION_COUNTRY"",null,null}, … {{3,2},{2,4},""HasRisk"",{0,6},null,null,null,null,null}]}"
"{{18,0},""Entity"",""sz:156"",""Rehana Badshah"",""sz:Person"",null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person""}","{{18,0},{4,0},""Related"",{503,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+NATIONALITY"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,""corp.disqual""}","{[{{126,0},""Entity"",null,""sz:2"",""LMAR GB LTD"",""sz:Organization"",null,null,null}, {{198,0},""Entity"",null,""sz:99"",""WELLHANCIA HEALTH CARE LTD"",""sz:Organization"",null,null,null}, … {{3,2},""OpenSanctions"",null,""sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ"",""Abassin BADSHAH"",""sz:Person"",""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",""https://www.opensanctions.org/entities/NK-25vyVFzt8vdJGgAXMRTwTJ"",null}],[{{4,0},{126,0},""Related"",{663,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(DIRECTORSHIP:)-RECORD_TYPE"",null,null}, {{126,0},{198,0},""Related"",{83,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+REGISTRATION_COUNTRY"",null,null}, … {{3,2},{2,4},""HasRisk"",{0,6},null,null,null,null,null}]}"
"{{193,0},""Entity"",""sz:9"",""BARLLOWS SERVICES LTD"",""sz:Organization"",null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person""}","{{193,0},{4,0},""Related"",{524,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(:DIRECTORSHIP)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,""corp.disqual""}","{[{{126,0},""Entity"",null,""sz:2"",""LMAR GB LTD"",""sz:Organization"",null,null,null}, {{198,0},""Entity"",null,""sz:99"",""WELLHANCIA HEALTH CARE LTD"",""sz:Organization"",null,null,null}, … {{3,2},""OpenSanctions"",null,""sz:ds_open-sanctions_NK-25vyVFzt8vdJGgAXMRTwTJ"",""Abassin BADSHAH"",""sz:Person"",""31 Quernmore Close, Bromley, Kent, United Kingdom, BR1 4EL"",""https://www.opensanctions.org/entities/NK-25vyVFzt8vdJGgAXMRTwTJ"",null}],[{{4,0},{126,0},""Related"",{663,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OPEN-SANCTIONS(DIRECTORSHIP:)-RECORD_TYPE"",null,null}, {{126,0},{198,0},""Related"",{83,1},""skos:related"",""POSSIBLY_RELATED"",""+ADDRESS+REGISTRATION_COUNTRY"",null,null}, … {{3,2},{2,4},""HasRisk"",{0,6},null,null,null,null,null}]}"
"{{198,0},""Entity"",""sz:99"",""WELLHANCIA HEALTH CARE LTD"",""sz:Organization"",null,null,null,null}","{{4,0},""Entity"",""sz:1"",""Abassin Badshah"",""sz:Person""}","{{198,0},{4,0},""Related"",{542,1},""skos:related"",""DISCLOSED"",""+ADDRESS+OOR(APPOINTMENT_OF_BOARD,SHAREHOLDING 75% 100%,VOTING_RIGHTS 75% 100%:)-RECORD_TYPE"",null,null}","{{2,4},""Risk"",null,null,null,null,null,null,""corp.disqual""}","{[{{126,0},""Entity"",null,""sz:2"",""LMAR GB LTD"",""sz:Organization"",null,null,null}, {{198,0},""Entity"",null,""sz:99"",""WELLHANCIA HEALTH CARE LTD"",""sz:Organization"",null,null,null}, … 

Finally, close the database connection.

In [8]:
db.close()

---